# 00 · 環境設定與第一次執行

這是概念軌的第 0 站。目標只有三個：

1. 確認 Gemini 金鑰與 ADK 版本都對
2. 跑出你的第一個 agent
3. **看懂 ADK 回傳的「事件串流」** ← 這一點才是本章真正的重點

第 3 點常被跳過，但它決定了你之後 debug 的能力。ADK 的 Runner 不會回傳一個
字串，它回傳的是一連串 `Event`。多 agent、工具呼叫、交棒，全部都在這條串流
裡面。看不懂它，後面每一章都會卡。

## 1. 環境檢查

這個系列需要 **ADK 2.x**。1.x 沒有 `Workflow` 圖形化執行引擎，第 10 章會整章跑不起來。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

# 讓 notebook 找得到專案根目錄的 shared/ 套件
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import quiet

quiet()  # 關掉幾個會洗版的第三方警告，細節見 shared/config.py

import google.adk

print(f"Python      : {sys.version.split()[0]}")
print(f"google-adk  : {google.adk.__version__}")

major = int(google.adk.__version__.split(".")[0])
assert major >= 2, "本教材需要 ADK 2.x，請執行 `uv sync` 重建環境"
print("✅ ADK 版本符合需求")

Python      : 3.13.15
google-adk  : 2.8.0
✅ ADK 版本符合需求


### 金鑰

`shared.require_api_key()` 會在金鑰缺失時給出可行動的錯誤訊息，
而不是讓你一路跑到 API 回 404 才發現。

金鑰來源優先序：`.env` 的 `GOOGLE_API_KEY` → shell 的 `GEMINI_API_KEY`。

In [2]:
from shared import load_settings, require_api_key

settings = load_settings()
require_api_key(settings)

print(f"provider : {settings.provider}")
print(f"model    : {settings.model_name}")
print(f"api_key  : {'已設定 (' + settings.api_key[:6] + '…)' if settings.api_key else '未設定'}")

provider : gemini
model    : gemini-flash-lite-latest
api_key  : 已設定 (AIzaSy…)


> **模型會下架。** `gemini-2.0-flash` 在 2026 年被停用，呼叫直接回 404 並要求
> 改用 `gemini-3.6-flash`。所以本教材把模型 ID 集中在 `shared/config.py` 的
> `DEFAULT_MODEL` 一個常數裡——真的哪天壞了，只要改一行。

## 2. 你的第一個 Agent

一個 `LlmAgent` 最少只需要三樣東西：

| 欄位 | 意義 |
|---|---|
| `name` | 識別用。多 agent 時，事件串流靠它分辨是誰在講話 |
| `model` | 背後接哪個 LLM |
| `instruction` | 系統提示，定義它的行為 |

In [3]:
from google.adk.agents import LlmAgent

from shared import get_model

greeter = LlmAgent(
    name="greeter",
    model=get_model(),
    instruction="你是一位親切的助理。用繁體中文回答，答案控制在兩句話以內。",
)

print(f"agent name : {greeter.name}")
print(f"model      : {greeter.model.model}")

agent name : greeter
model      : gemini-flash-lite-latest


注意 `get_model()` 回傳的不是字串，而是一個 **`Gemini` 物件**。

`LlmAgent(model="gemini-2.5-flash")` 這種字串寫法也可以，而且更短。但字串沒
有地方掛「重試設定」——免費層很容易撞到 429（配額）和 503（模型忙碌），
沒有自動重試，一本 notebook 跑到一半就會斷。這是「模型字串 vs 模型物件」
的第一個實際差異，第 03 章會完整比較。

## 3. 跑起來：Runner 與事件串流

Agent 本身只是「設定」，不會自己動。真正執行它的是 **Runner**。

```
你的問題 ──▶ Runner ──▶ Agent ──▶ 模型
                │
                └──▶ 吐出一連串 Event ──▶ 你
```

`InMemoryRunner` 是開發用的簡化版，session／artifact／memory 全部放在記憶體，
重開 kernel 就清空。正式環境會換成有持久化的服務（第 04 章）。

In [4]:
from google.adk.runners import InMemoryRunner
from google.genai import types

runner = InMemoryRunner(agent=greeter, app_name="concept_track")

session = await runner.session_service.create_session(
    app_name="concept_track", user_id="student"
)

message = types.Content(role="user", parts=[types.Part(text="你好，用一句話介紹你自己")])

async for event in runner.run_async(
    user_id="student", session_id=session.id, new_message=message
):
    print(f"[{event.author}] final={event.is_final_response()}")
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(f"    {part.text.strip()}")

[greeter] final=True
    你好！我是親切的助理，隨時準備好為您提供協助。


### 事件解剖

每個 `Event` 身上你會反覆用到的欄位：

| 欄位 | 用途 |
|---|---|
| `event.author` | 誰產生的。多 agent 時就是 agent 名稱 |
| `event.content.parts` | 內容。一個 event 可能有多個 part |
| `part.text` | 文字 |
| `part.function_call` | 模型「想呼叫工具」 |
| `part.function_response` | 工具「執行完回傳」 |
| `part.thought` | `True` 代表這是模型的推理過程，**不該顯示給使用者** |
| `event.is_final_response()` | 這是不是這一輪的最終答覆 |

`part.thought` 這個坑值得特別記：thinking 模型的最終回應會包含多個 part，
如果你直接 `parts[0].text` 印出來，使用者會看到模型的自言自語。

## 4. 把樣板收起來

上面那段 `async for` 每個 cell 都寫一次會把重點淹沒。`shared/runtime.py`
已經包好了，之後的章節直接用：

| 函式 | 用途 |
|---|---|
| `run_once(agent, text)` | 一次性問答，自己建 runner 和 session |
| `new_session(runner)` / `ask(runner, text, session_id=...)` | 多輪對話 |
| `ask(..., trace=True)` | 順便印出工具呼叫與交棒過程 |
| `peek_state(runner, session_id)` | 看 session state 裡有什麼 |

In [5]:
from shared import ask, new_session, run_once

answer = await run_once(greeter, "用一句話解釋什麼是 AI Agent")
print(answer)

AI Agent 就像是能自主思考並採取行動的數位助手，不僅能回答問題，還能幫你完成複雜的任務。


### 多輪對話

同一個 `session_id` 就是同一段對話。第二個問題裡的「它」能被理解，
是因為 session 保留了上一輪的內容。

In [6]:
chat = InMemoryRunner(agent=greeter, app_name="concept_track")
sid = await new_session(chat)

print("Q1:", await ask(chat, "我叫 Sean，正在學 Google ADK。", session_id=sid))
print("Q2:", await ask(chat, "我剛剛說我在學什麼？", session_id=sid))

Q1: 哈囉 Sean！很高興認識你，祝你學習 Google ADK 的過程順利又愉快！


Q2: 你剛才提到你正在學習 Google ADK。如果有任何相關問題，隨時都可以問我喔！


## 5. 加一個工具，看清楚事件串流

工具是 agent 跟外界互動的手腳。這裡先看它在事件串流裡長什麼樣子——
完整的工具設計留到第 02 章。

打開 `trace=True`，你會看到一次工具呼叫其實是 **兩個** 事件：
模型先發出 `function_call`，ADK 執行完再回填 `function_response`。

In [7]:
def add_numbers(a: int, b: int) -> dict:
    """把兩個整數相加。

    Args:
        a: 第一個數字。
        b: 第二個數字。
    """
    return {"result": a + b}


calculator = LlmAgent(
    name="calculator",
    model=get_model(),
    instruction="你是計算助理。遇到算術問題一定要呼叫 add_numbers 工具，不要自己心算。",
    tools=[add_numbers],
)

result = await run_once(calculator, "幫我算 1234 加 5678", trace=True)
print("\n最終答覆:", result)

  🔧 [calculator] 呼叫 add_numbers({'b': 5678, 'a': 1234})
  ↩️  [calculator] add_numbers 回傳 {'result': 6912}


  💬 [calculator] 1234 加 5678 的結果是 6912。

最終答覆: 1234 加 5678 的結果是 6912。


## 6. 省錢與省麻煩：RateLimiter

AI Studio 免費層是用「每分鐘請求數（RPM）」計費的。而一個多 agent 的 cell
可能一口氣送出十幾次請求，很容易撞到 429。

`shared/plugins.py` 提供了一個 `RateLimiter` **Plugin**，掛在 Runner 上就
全域生效。Plugin 的完整機制在第 06 章，這裡先當成一個實用工具用。

In [8]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from shared import CallCounter, RateLimiter

counter = CallCounter()

limited = Runner(
    agent=calculator,
    app_name="concept_track",
    session_service=InMemorySessionService(),
    plugins=[RateLimiter(rpm=10), counter],
)

sid2 = await new_session(limited)
print(await ask(limited, "算 99 加 1", session_id=sid2))
print("\n📊", counter.report())

99 加 1 的結果是 100。

📊 模型呼叫 2 次；工具呼叫 1 次：add_numbers


### 撞到 429 的時候怎麼辦

免費層一定會遇到 `429 RESOURCE_EXHAUSTED`。有兩件事值得知道：

**1. 配額是「每個模型各自計算」的。**
`gemini-2.5-flash` 用完了，`gemini-2.5-flash-lite` 還是照跑——不必等到隔天。
所以撞牆時第一個動作是換模型 ID，不是去睡覺。

In [9]:
from shared import FALLBACK_MODELS, pick_available_model

print("候補模型清單：")
for name in FALLBACK_MODELS:
    print(f"  {name}")

print("\n實際探測哪一個還有配額：")
usable = pick_available_model()
print(f"\n→ 可以用 ADK_MODEL={usable}")

候補模型清單：
  gemini-flash-lite-latest
  gemini-3.5-flash-lite
  gemini-3.1-flash-lite
  gemini-flash-latest
  gemini-3.5-flash
  gemini-2.5-flash-lite
  gemini-2.5-flash

實際探測哪一個還有配額：


✅ gemini-flash-lite-latest 可用

→ 可以用 ADK_MODEL=gemini-flash-lite-latest


**2. 本教材已經內建節流。**
`shared/runtime.py` 在 `ask()` 裡放了一道全域閘門（預設 12 RPM），
因為所有提問最後都會經過 `ask()`，在那裡設限最省事。
想關掉就設環境變數 `ADK_RPM=0`。

這跟上面 `RateLimiter` plugin 的差別：plugin 攔的是「模型呼叫」（更精準，
一次工具呼叫算兩次），全域閘門攔的是「你問了幾個問題」（更簡單）。
兩者可以並用。

`CallCounter` 印出的數字值得看一眼：**一次工具呼叫會用掉兩次模型呼叫**。
第一次模型決定「要呼叫 add_numbers」，第二次模型拿到工具結果後才產生答案。
這就是為什麼多 agent 系統的成本會比你直覺估計的高。

## 本章重點

- **Agent 是設定，Runner 才是執行者。** Agent 物件本身不會動。
- **Runner 回傳事件串流，不是字串。** `author` / `function_call` /
  `function_response` / `thought` 是你之後 debug 的全部依據。
- **`part.thought=True` 的內容不該給使用者看。**
- **一次工具呼叫 = 兩次模型呼叫。** 成本要這樣算。
- 樣板都收在 `shared/`：`run_once`、`ask`、`new_session`、`peek_state`。
- **免費層配額是每個模型分開算的**，撞到 429 先換模型 ID。

## 動手練習

1. 把 `greeter` 的 `instruction` 改成「只能用英文回答」，重跑第 3 節，
   確認 instruction 真的有效。
2. 在第 5 節的 `add_numbers` 裡加一行 `print("工具被呼叫了！")`，
   重跑並觀察它出現在事件串流的哪個位置。
3. 把 `calculator` 的 instruction 改成「可以自己心算」，看模型還會不會
   呼叫工具。（提示：`CallCounter` 的 `tool_calls` 會變空）

---
**下一站 → `01_agent_basics.ipynb`**：Agent 的三個組成要素，
以及為什麼 `description` 不是註解。